In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [9]:
# データ読み込み（raw）
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']

print('Train:', train.shape)
print('X:', X.shape, 'y:', y.shape)
print('Target mean:', float(y.mean()))

Train: (700000, 26)
X: (700000, 24) y: (700000,)
Target mean: 0.6232957142857143


In [10]:
# CV前提でシンプル比較（これまでの流れに合わせる）
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Categorical cols:', cat_cols)
print('Numeric cols:', len(num_cols))
print('CV:', cv)

Categorical cols: ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']
Numeric cols: 18
CV: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [11]:
def cv_eval_model(name, model, X, y, cv):
    scoring = {
        'auc': 'roc_auc',
        'acc': 'accuracy',
    }
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    return {
        'model': name,
        'auc_mean': float(np.mean(scores['test_auc'])),
        'auc_std': float(np.std(scores['test_auc'])),
        'acc_mean': float(np.mean(scores['test_acc'])),
        'acc_std': float(np.std(scores['test_acc'])),
    }

In [ ]:
# XGBoost / LightGBM 用：最小限のOrdinal Encoding（raw特徴を壊さないため）
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
 )

# これまでの流れに合わせて「最小限」：主にrandom_stateだけ固定（比較の再現性）
xgb = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', XGBClassifier(
        random_state=42,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric='logloss',
    )),
 ])

lgbm = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LGBMClassifier(
        random_state=42,
    )),
 ])

results = []
results.append(cv_eval_model('xgboost', xgb, X, y, cv))
results.append(cv_eval_model('lightgbm', lgbm, X, y, cv))

pd.DataFrame(results).sort_values('auc_mean', ascending=False)

[LightGBM] [Info] Number of positive: 349046, number of negative: 210954
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050402 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1648
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623296 -> initscore=0.503564
[LightGBM] [Info] Start training from score 0.503564


c:\Users\mikku\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,auc,acc
1,lightgbm,0.725079,0.683336
0,xgboost,0.724691,0.683557


In [ ]:
# CatBoost（カテゴリ列をrawで扱える）
try:
    from catboost import CatBoostClassifier

    def cv_eval_catboost(X, y, cat_cols, cv):
        cat_idx = [X.columns.get_loc(c) for c in cat_cols]
        aucs = []
        accs = []
        for tr_idx, va_idx in cv.split(X, y):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            model = CatBoostClassifier(
                iterations=300,
                depth=6,
                learning_rate=0.1,
                loss_function='Logloss',
                eval_metric='AUC',
                random_seed=42,
                verbose=False,
            )
            model.fit(X_tr, y_tr, cat_features=cat_idx)
            proba = model.predict_proba(X_va)[:, 1]
            pred = (proba >= 0.5).astype(int)
            aucs.append(roc_auc_score(y_va, proba))
            accs.append(accuracy_score(y_va, pred))

        return {
            'model': 'catboost',
            'auc_mean': float(np.mean(aucs)),
            'auc_std': float(np.std(aucs)),
            'acc_mean': float(np.mean(accs)),
            'acc_std': float(np.std(accs)),
        }

    results.append(cv_eval_catboost(X, y, cat_cols, cv))
    pd.DataFrame(results).sort_values('auc_mean', ascending=False)
except Exception as e:
    print('CatBoost skipped:', repr(e))
    print('Install with: pip install catboost')

CatBoost skipped: ModuleNotFoundError("No module named 'catboost'")
Install with: pip install catboost


## 次（アンサンブル）

このノートで上位モデルが決まったら、
- 予測確率の単純平均（soft-vote）
- 重み付き平均
をまず1セルで試します。